In [3]:
import pandas as pd
import numpy as np

# Load data and convert date to datetime format
df = pd.read_csv('/train.csv')
df['date'] = pd.to_datetime(df['date'])

# To save time in this 90-minute window, we will subset the data to just Store 1
df = df[df['store'] == 1].copy()

In [4]:
# 1. Seasonality features
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

# Sort chronologically to ensure past data predicts future data
df = df.sort_values(by=['item', 'date'])

# 2. Lags and Rolling Stats (Engineer time-based features)
# Shift(7) looks at the sales from exactly 7 days ago
df['lag_7'] = df.groupby('item')['sales'].shift(7)

# Rolling mean calculates the average sales over the previous 7 days
df['rolling_mean_7'] = df.groupby('item')['sales'].transform(lambda x: x.shift(1).rolling(7).mean())

# Drop rows with NaN values created by lags
df = df.dropna()

# Split into train (past) and test (future)
train = df[df['date'] < '2017-01-01']
test = df[df['date'] >= '2017-01-01']

features = ['day_of_week', 'month', 'lag_7', 'rolling_mean_7']
X_train, y_train = train[features], train['sales']
X_test, y_test = test[features], test['sales']

In [5]:
import lightgbm as lgb

# Train 50th Percentile (Point Forecast / Median)
model_50 = lgb.LGBMRegressor(objective='quantile', alpha=0.5)
model_50.fit(X_train, y_train)
pred_50 = model_50.predict(X_test)

# Train 10th Percentile (Lower bound)
model_10 = lgb.LGBMRegressor(objective='quantile', alpha=0.1)
model_10.fit(X_train, y_train)
pred_10 = model_10.predict(X_test)

# Train 90th Percentile (Upper bound / Safety Stock level)
model_90 = lgb.LGBMRegressor(objective='quantile', alpha=0.9)
model_90.fit(X_train, y_train)
pred_90 = model_90.predict(X_test)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001594 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 412
[LightGBM] [Info] Number of data points in the train set: 72700, number of used features: 4
[LightGBM] [Info] Start training from score 43.000000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003529 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 412
[LightGBM] [Info] Number of data points in the train set: 72700, number of used features: 4
[LightGBM] [Info] Start training from score 18.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001150 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins

In [6]:
from sklearn.metrics import mean_absolute_error

# Evaluate Point Accuracy (MAE)
mae = mean_absolute_error(y_test, pred_50)
print(f"Mean Absolute Error (Point Forecast): {mae:.2f}")

# Evaluate Interval Calibration
# Check if actual sales are between our 10th and 90th predictions
within_interval = (y_test >= pred_10) & (y_test <= pred_90)
coverage = within_interval.mean() * 100
print(f"Interval Calibration (Coverage): {coverage:.2f}% (Target is ~80%)")

Mean Absolute Error (Point Forecast): 6.28
Interval Calibration (Coverage): 78.91% (Target is ~80%)


In [7]:
# Translate the interval into a decision: Cost Tradeoff
stockout_cost = 10
holding_cost = 2

def calculate_cost(predictions, actuals):
    # If actual demand > predicted, we stock out
    stockouts = np.maximum(0, actuals - predictions) * stockout_cost
    # If predicted > actual demand, we overstock
    overstocks = np.maximum(0, predictions - actuals) * holding_cost
    return np.sum(stockouts + overstocks)

cost_point_forecast = calculate_cost(pred_50, y_test)
cost_safety_stock = calculate_cost(pred_90, y_test)

print(f"Total Cost using Point Forecast (50th): ${cost_point_forecast:,.2f}")
print(f"Total Cost using Safety Stock (90th): ${cost_safety_stock:,.2f}")
print("Conclusion: The 90th percentile minimizes stockouts, significantly reducing overall asymmetric costs.")

Total Cost using Point Forecast (50th): $744,293.94
Total Cost using Safety Stock (90th): $456,395.03
Conclusion: The 90th percentile minimizes stockouts, significantly reducing overall asymmetric costs.
